In [1]:
import pandas as pd
import numpy as np

# Загружаем наши очищенные данные
profile = pd.read_parquet('../data/processed/profile_clean.parquet')
transcript = pd.read_parquet('../data/processed/transcript_clean.parquet')
portfolio = pd.read_parquet('../data/processed/portfolio.parquet')

In [2]:
# Оставляем только полученные офферы
offers_received = transcript[transcript['event'] == 'offer received'].copy()

# Добавляем информацию о длительности оффера (duration) из портфолио
offers_received = offers_received.merge(portfolio[['id', 'duration']], left_on='offer_id', right_on='id', how='left')

# Переведем duration в минуты (если time в данных в минутах, а duration в днях)
# В датасете Udacity duration указан в днях (например, 7 дней). 
# Время (time) в транскрипте тоже в днях (0, 1, 2... 30). Проверим это.
print(offers_received[['person', 'time', 'duration']].head())

                             person  time  duration
0  78afa995795e4d85b5d9ceeca43f5fef     0         7
1  a03223e636434f42ac4c3df47e8bac43     0        10
2  e2127556f4f64592b11af22de27a7932     0         7
3  8ec6ce2a7e7949b1bf142def7d0e0586     0        10
4  68617ca6246f4fbc85e91a2a49552598     0         5


In [3]:
transactions = transcript[transcript['event'] == 'transaction'].copy()
# Нам нужны только время и пользователь. Сумма amount нам сейчас не важна, важна частота
transactions = transactions[['person', 'time', 'amount']]

In [4]:
# Соединяем офферы и транзакции по пользователю
# Получится таблица, где у каждого оффера будет список ВСЕХ транзакций пользователя
merged = offers_received.merge(transactions, on='person', suffixes=('_offer', '_trans'))

# Теперь фильтруем: оставляем только те транзакции, которые произошли ПОСЛЕ получения оффера
# И до окончания срока действия оффера (time_offer + duration)
converted = merged[
    (merged['time_trans'] >= merged['time_offer']) & 
    (merged['time_trans'] <= merged['time_offer'] + merged['duration'])
]

# Теперь нам нужно понять: сколько офферов привели к конверсии, а сколько нет.
# Для этого создадим колонку is_converted = 1 для тех, кто попал в фильтр выше.
converted['is_converted'] = 1

In [5]:
# Берем список всех полученных офферов
final_df = offers_received[['person', 'offer_id', 'time', 'duration']].copy()

# Мержим с конверсиями (left join, чтобы сохранить всех, у кого не было конверсии)
# Но нам нужно избежать дублирования, если было несколько транзакций (берем уникальные)
conversion_status = converted[['person', 'offer_id', 'is_converted']].drop_duplicates(subset=['person', 'offer_id'])

final_df = final_df.merge(conversion_status, on=['person', 'offer_id'], how='left')

# Заполняем пропуски нулями (если не было конверсии)
final_df['is_converted'] = final_df['is_converted'].fillna(0).astype(int)

print("Размер итоговой таблицы:", final_df.shape)
print("Средняя конверсия:", final_df['is_converted'].mean().round(4))

Размер итоговой таблицы: (76277, 5)
Средняя конверсия: 0.1442


In [6]:
# 1. Подтягиваем демографию из профиля
final_df = final_df.merge(profile[['id', 'age', 'gender', 'income']], left_on='person', right_on='id', how='left')

# 2. Создаем возрастные группы
# Используем те же границы, что и в KKBox, для консистентности
final_df['age_group'] = pd.cut(final_df['age'], bins=[0, 25, 35, 50, 100], labels=['<25', '25-35', '35-50', '50+'])

# 3. Считаем конверсию по возрастным группам
conversion_by_age = final_df.groupby('age_group', observed=True)['is_converted'].agg(['mean', 'count']).reset_index()
print("Конверсия по возрастным группам:")
print(conversion_by_age)

Конверсия по возрастным группам:
  age_group      mean  count
0       <25  0.167290   4543
1     25-35  0.177943   6193
2     35-50  0.152503  14780
3       50+  0.135491  40962


In [7]:
# Подтягиваем тип оффера
final_df = final_df.merge(portfolio[['id', 'offer_type']], left_on='offer_id', right_on='id', how='left')

# Смотрим конверсию по типу оффера и возрасту
pivot_table = final_df.pivot_table(
    index='age_group', 
    columns='offer_type', 
    values='is_converted', 
    aggfunc='mean'
)
print(pivot_table)

offer_type      bogo  discount  informational
age_group                                    
<25         0.163446  0.205338       0.097098
25-35       0.174899  0.211531       0.114238
35-50       0.154766  0.176491       0.100169
50+         0.126931  0.169693       0.084285


In [8]:
import statsmodels.formula.api as smf

# 1. Для удобства сделаем бинарный признак для скидки (discount = 1, остальные = 0)
final_df['is_discount'] = (final_df['offer_type'] == 'discount').astype(int)

# 2. Создадим бинарный признак для молодой группы (25-35)
# (Мы сравниваем 25-35 с самой большой группой 50+, чтобы модель была стабильной)
final_df['is_young'] = (final_df['age_group'] == '25-35').astype(int)

# 3. Строим логистическую регрессию с interaction term
# Формула: конверсия ~ скидка + молодость + (скидка * молодость)
model = smf.logit('is_converted ~ is_discount + is_young + is_discount:is_young', data=final_df).fit()

# Выводим результат
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.409429
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:           is_converted   No. Observations:                76277
Model:                          Logit   Df Residuals:                    76273
Method:                           MLE   Df Model:                            3
Date:                Fri, 23 Oct 2026   Pseudo R-squ.:                0.007530
Time:                        13:27:36   Log-Likelihood:                -31230.
converged:                       True   LL-Null:                       -31467.
Covariance Type:            nonrobust   LLR p-value:                2.183e-102
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               -1.9919      0.015   -132.771      0.000      -2.021      -1.963